# Remake pathological heating validation figure for the paper.

Replaces analytic GF with CosmoTherm GF convolution.
Three panels: sinusoidal, wide Gaussian, double power-law.
Each panel: PDE (solid), PDE GF table (dashed), CosmoTherm GF (dotted).

In [2]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from spectroxide import (
    g_bb, delta_n_to_delta_I, decompose_distortion, apply_style,
)
from spectroxide.solver import run_sweep
from spectroxide.greens_table import load_or_build_greens_table
from spectroxide.cosmotherm import load_greens_database, convolve_cosmotherm_gf
from spectroxide.plot_params import (
    C, DOUBLE_COL, LW, LW_THICK, LW_THIN, LEGEND_SIZE,
)

apply_style()

## Helpers

In [3]:
def strip_gbb(x, delta_n):
    """NC strip: subtract G_bb so int x^2 Dn dx = 0."""
    gbb = g_bb(x)
    alpha = np.trapz(x**2 * delta_n, x) / np.trapz(x**2 * gbb, x)
    return delta_n - alpha * gbb, alpha


def rms_percent(nu1, di1, nu2, di2, nu_lo=30, nu_hi=800):
    """RMS % between two spectra, interpolated to common grid."""
    mask = (nu1 >= nu_lo) & (nu1 <= nu_hi)
    di2_interp = np.interp(nu1, nu2, di2)
    ref = np.max(np.abs(di2_interp[mask]))
    if ref < 1e-50:
        return np.nan
    return np.sqrt(np.mean((di1[mask] - di2_interp[mask])**2)) / ref * 100

## Heating scenarios

In [ ]:
k_sin = 2 * np.pi / 5e4
A_sin = 5e-10

def dq_dz_sin(z):
    if z < 1e3 or z > 5e5:
        return 0.0
    return A_sin * np.sin(k_sin * (1 + z))

z0_gauss = 8e4
sigma_gauss = 4e4
A_gauss = 1e-5 / (sigma_gauss * np.sqrt(2 * np.pi))

def dq_dz_gauss(z):
    return A_gauss * np.exp(-0.5 * ((z - z0_gauss) / sigma_gauss)**2)

z_c = 3e5
A_dpl = 1e-5 / (6 * z_c**4)

def dq_dz_dpl(z):
    return A_dpl * z**3 * np.exp(-z / z_c)

scenarios = [
    ("Sinusoidal heating/cooling",              dq_dz_sin,   C["red"]),
    (r"Wide Gaussian ($\mu$--$y$ transition)",   dq_dz_gauss, C["blue"]),
    (r"Peaked power-law ($z > 10^6$)",           dq_dz_dpl,   C["orange"]),
]

## Load tables

In [5]:
print("Loading PDE GF table (production quality)...", flush=True)
_HQ_CACHE = Path.home() / ".spectroxide" / "greens_table_hq.npz"
heat_table = load_or_build_greens_table(
    cache_path=_HQ_CACHE,
    z_injections=np.logspace(np.log10(1e3), np.log10(5e6), 200),
    n_points=4000,
    n_x=4000,
    x_min=0.005,
    x_max=40.0,
    z_end=100,
    timeout=3600,
)
print(f"  {len(heat_table.z_h)} redshifts, {len(heat_table.x)} freq pts")

print("Loading CosmoTherm GF database...", flush=True)
z_h_ct, x_ct, g_th_ct = load_greens_database()
print(f"  {len(z_h_ct)} redshifts x {len(x_ct)} frequencies")

Loading PDE GF table (production quality)...


  200 redshifts, 4000 freq pts
Loading CosmoTherm GF database...


  118 redshifts x 4190 frequencies


## Run comparisons

In [6]:
x_obs = np.linspace(0.1, 25, 1000)
NU_TO_X = 1.0 / 56.786  # x = nu_GHz * NU_TO_X

results = []
for name, dq_fn, color in scenarios:
    print(f"\n=== {name} ===")

    # --- PDE ---
    print("  Running PDE...", flush=True)
    pde_result = run_sweep(
        dq_dz=dq_fn,
        method="pde",
        z_min=1e3,
        z_max=3e6,
        z_end=100,
        n_z=10000,
        n_points=8000,
        number_conserving=True,
        timeout=1200,
    )
    x_pde = np.array(pde_result["results"][0]["x"])
    dn_pde = np.array(pde_result["results"][0]["delta_n"])
    dn_pde_nc, alpha_pde = strip_gbb(x_pde, dn_pde)
    nu_pde, di_pde = delta_n_to_delta_I(x_pde, dn_pde_nc)
    dec_pde = decompose_distortion(x_pde, dn_pde_nc)
    print(f"  PDE: mu={dec_pde['mu']:.3e}, y={dec_pde['y']:.3e}")

    # --- PDE GF Table ---
    print("  GF table convolution...", flush=True)
    dn_tab = heat_table.distortion_from_heating(x_obs, dq_fn, 1e3, 3e6, n_z=5000)
    dn_tab_nc, alpha_tab = strip_gbb(x_obs, dn_tab)
    nu_tab, di_tab = delta_n_to_delta_I(x_obs, dn_tab)
    dec_tab = decompose_distortion(x_obs, dn_tab_nc)
    print(f"  Table: mu={dec_tab['mu']:.3e}, y={dec_tab['y']:.3e}")

    # --- CosmoTherm GF ---
    print("  CosmoTherm GF convolution...", flush=True)
    x_ct_out, di_ct_jy = convolve_cosmotherm_gf(
        z_h_ct, x_ct, g_th_ct, dq_fn,
        z_min=1e3, z_max=3e6, n_z=5000,
    )
    _K_B = 1.380_649e-23
    _H_PL = 6.626_070_15e-34
    _T_CMB = 2.726
    nu_ct = x_ct_out * _K_B * _T_CMB / _H_PL / 1e9  # GHz

    rms_tab = rms_percent(nu_tab, di_tab, nu_pde, di_pde)
    rms_ct = rms_percent(nu_ct, di_ct_jy, nu_pde, di_pde)
    print(f"  RMS vs PDE: table={rms_tab:.1f}%, CosmoTherm={rms_ct:.1f}%")

    results.append({
        "name": name, "color": color,
        "nu_pde": nu_pde, "di_pde": di_pde,
        "nu_tab": nu_tab, "di_tab": di_tab,
        "x_ct": x_ct_out, "di_ct": di_ct_jy,
        "rms_tab": rms_tab, "rms_ct": rms_ct,
    })


=== Sinusoidal ===
  Running PDE...


  PDE: mu=-5.536e-06, y=9.771e-07
  GF table convolution...


  Table: mu=-5.546e-06, y=9.826e-07
  CosmoTherm GF convolution...


  RMS vs PDE: table=0.3%, CosmoTherm=0.4%

=== Wide Gaussian ===
  Running PDE...


  PDE: mu=1.172e-05, y=1.003e-06
  GF table convolution...


  Table: mu=1.170e-05, y=1.008e-06
  CosmoTherm GF convolution...


## Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, 3.0))

for idx, hr in enumerate(results):
    ax = axes[idx]
    ax.plot(hr["nu_pde"] * NU_TO_X, hr["di_pde"], color=hr["color"], lw=LW_THICK, label="PDE")
    ax.plot(hr["nu_tab"] * NU_TO_X, hr["di_tab"], color="k", lw=LW_THIN, ls="--", label="GF table")
    ax.plot(hr["x_ct"], hr["di_ct"], color="gray", lw=LW_THIN, ls=":", label=r"\textsc{CosmoTherm} GF")
    ax.set_xlabel(r"$x$")
    ax.set_title(hr["name"], fontsize=8)
    ax.set_xlim(0, 20)
    if idx == 0:
        ax.set_ylabel(r"$\Delta I$ [Jy/sr]")
        ax.legend(fontsize=LEGEND_SIZE - 1, loc='best')

plt.tight_layout()
outpath = "../../notebooks/figures/pathological_heating_validation.pdf"
fig.savefig(outpath, bbox_inches="tight")
print(f"\nSaved {outpath}")
plt.close(fig)